# HyConEx + HyperLogic (DR-HyperCF) sur les jeux HyConEx / HyperLogic

Entraînement du modèle hybride (TabResNet HyConEx + règles / CF HyperLogic) calibré sur Dry Bean.

**Sources de données**
- `HyConEx/data/` : adult, audit, blobs, compas, german_credit, heloc, law, moons, moons_with_blob, wine
- `HyperLogic/data/` : cardio, disease, brca-n, brca-s

**Prétraitement**
- colonnes **continues** → MinMax + intervalles (quantiles) → bipolarisation
- colonnes **catégorielles** → one-hot → bipolarisation

Pour **chaque jeu** : métriques, contrefactuels et règles extraites.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nouveau_module import HybridDRConfig, HybridDRTrainer
from hyconex_hyperlogic_datasets import list_available_datasets
from tabular_mixed_preprocessing import TabularDatasetSplits, get_dataset_loaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Racine projet:', ROOT.parent)
print('Jeux détectés:', list_available_datasets())

Device: cuda
Racine projet: D:\ecole\master 2\recherche\INN\projet
Jeux détectés: {'hyconex': ['adult', 'audit', 'blobs', 'compas', 'german_credit', 'heloc', 'law', 'moons', 'wine', 'moons_with_blob'], 'hyperlogic': ['cardio', 'disease', 'brca-n', 'brca-s']}


## Configuration

Par défaut, tous les CSV/TSV présents localement sont entraînés.
Tu peux restreindre la liste `DATASETS` ou activer `QUICK_RUN`.

In [2]:
ALL_LOADERS = get_dataset_loaders()

# None = tous les jeux disponibles ; sinon liste explicite, ex. ['hyconex/adult', 'hyperlogic/cardio']
DATASETS = None

if DATASETS is None:
    DATASETS = sorted(ALL_LOADERS.keys())

SEED = 42
QUICK_RUN = False
BASE_EPOCHS = 8 if QUICK_RUN else 40
LARGE_EPOCHS = 6 if QUICK_RUN else 25

print(f'{len(DATASETS)} jeux sélectionnés:', DATASETS)

14 jeux sélectionnés: ['hyconex/adult', 'hyconex/audit', 'hyconex/blobs', 'hyconex/compas', 'hyconex/german_credit', 'hyconex/heloc', 'hyconex/law', 'hyconex/moons', 'hyconex/moons_with_blob', 'hyconex/wine', 'hyperlogic/brca-n', 'hyperlogic/brca-s', 'hyperlogic/cardio', 'hyperlogic/disease']


In [ ]:
def make_config(splits: TabularDatasetSplits) -> HybridDRConfig:
    n_classes = len(splits.class_names)
    n_feat = splits.X_train.shape[1]
    large = n_feat > 200 or n_classes > 8
    epochs = LARGE_EPOCHS if large else BASE_EPOCHS

    num_rules = 64
    batch = 128
    if n_feat > 500:
        num_rules = 48
        batch = 256
    if n_classes > 20:
        num_rules = min(96, max(48, n_classes * 2))
    if splits.name in ('moons', 'blobs', 'wine', 'law'):
        num_rules = 24
        batch = 32

    return HybridDRConfig(
        seed=SEED,
        epochs=epochs,
        batch_size=batch,
        lr=1e-3,
        num_rules=num_rules,
        bins_per_feature=4,
        hyper_hidden_dim=128,
        tabresnet_n_blocks=4,
        tabresnet_dropout=0.1,
        cf_lambda=0.15,
        flip_lambda=0.06,
        rule_sparsity_lambda=0.002,
        temperature=0.8,
        input_encoding=splits.input_encoding,
    )


def train_on_dataset(key: str) -> dict:
    splits = ALL_LOADERS[key]()
    cfg = make_config(splits)
    trainer = HybridDRTrainer(cfg, device=device)

    print(f"\n{'=' * 72}\n{key} | train {splits.X_train.shape} | classes {len(splits.class_names)} | enc={splits.input_encoding}\n{'=' * 72}")

    result = trainer.fit(
        splits.X_train,
        splits.y_train,
        x_val_cont=splits.X_val,
        y_val=splits.y_val,
        feature_names=splits.feature_names,
        class_names=splits.class_names,
        verbose=True,
    )
    metrics = trainer.evaluate(splits.X_test, splits.y_test, counterfactuals=True)
    rules = trainer.export_rules(top_per_rule=4, min_abs_weight=0.05)

    return {
        'key': key,
        'name': splits.name,
        'splits': splits,
        'trainer': trainer,
        'result': result,
        'metrics': metrics,
        'rules': rules,
    }


def metrics_row(run: dict) -> dict:
    m = run['metrics']
    cf = m.get('counterfactuals', {})
    return {
        'dataset': run['key'],
        'best_val_acc': run['result'].best_val_accuracy,
        'test_acc': m.get('accuracy'),
        'test_auroc': m.get('auroc_ovr'),
        'cf_validity': cf.get('validity_cf'),
        'cf_changed_bits': cf.get('changed_bits_mean'),
        'cf_l1_cont': cf.get('proximity_l1_cont_mean'),
        'n_rules': len(run['rules']),
        'n_binary_features': run['trainer'].binarizer.n_binary_features_,
    }

## Entraînement

In [4]:
runs: dict[str, dict] = {}
failures: dict[str, str] = {}

for ds_key in DATASETS:
    try:
        runs[ds_key] = train_on_dataset(ds_key)
    except FileNotFoundError as exc:
        failures[ds_key] = str(exc)
        print('SKIP', ds_key, '→', exc)
    except Exception as exc:
        failures[ds_key] = str(exc)
        print('ERREUR', ds_key, '→', exc)

if runs:
    summary_df = pd.DataFrame([metrics_row(r) for r in runs.values()])
    display(summary_df.round(4))
else:
    print('Aucun jeu entraîné. Vérifiez HyConEx/data et HyperLogic/data.')

if failures:
    print('\nJeux ignorés:')
    for k, v in failures.items():
        print('-', k, ':', v)


hyconex/adult | train (20838, 34) | classes 2 | enc=bipolar
[Epoch 001/40] loss=0.6204 val_acc=0.8071 val_auroc=0.8739 best(accuracy)=0.8071
[Epoch 002/40] loss=0.5278 val_acc=0.7681 val_auroc=0.8771 best(accuracy)=0.8071
[Epoch 003/40] loss=0.5149 val_acc=0.7449 val_auroc=0.8808 best(accuracy)=0.8071
[Epoch 004/40] loss=0.5050 val_acc=0.7541 val_auroc=0.8802 best(accuracy)=0.8071
[Epoch 005/40] loss=0.4989 val_acc=0.7704 val_auroc=0.8793 best(accuracy)=0.8071
[Epoch 006/40] loss=0.4998 val_acc=0.7729 val_auroc=0.8806 best(accuracy)=0.8071
[Epoch 007/40] loss=0.4947 val_acc=0.8048 val_auroc=0.8805 best(accuracy)=0.8071
[Epoch 008/40] loss=0.4941 val_acc=0.7871 val_auroc=0.8810 best(accuracy)=0.8071
[Epoch 009/40] loss=0.4881 val_acc=0.7678 val_auroc=0.8788 best(accuracy)=0.8071
[Epoch 010/40] loss=0.4878 val_acc=0.7846 val_auroc=0.8803 best(accuracy)=0.8071
[Epoch 011/40] loss=0.4845 val_acc=0.7631 val_auroc=0.8792 best(accuracy)=0.8071
[Epoch 012/40] loss=0.5256 val_acc=0.7797 val_au

D:\ecole\master 2\recherche\INN\projet\HyConEx\counterfactuals\datasets\compas.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  raw_data["length_of_stay"].fillna(
D:\ecole\master 2\recherche\INN\projet\HyConEx\counterfactuals\datasets\compas.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh


hyconex/compas | train (2693, 25) | classes 3 | enc=bipolar
[Epoch 001/40] loss=1.3345 val_acc=0.5475 val_auroc=0.7311 best(accuracy)=0.5475
[Epoch 002/40] loss=1.1685 val_acc=0.5623 val_auroc=0.7542 best(accuracy)=0.5623
[Epoch 003/40] loss=1.1382 val_acc=0.5564 val_auroc=0.7509 best(accuracy)=0.5623
[Epoch 004/40] loss=1.1114 val_acc=0.5668 val_auroc=0.7520 best(accuracy)=0.5668
[Epoch 005/40] loss=1.1022 val_acc=0.5549 val_auroc=0.7433 best(accuracy)=0.5668
[Epoch 006/40] loss=1.1014 val_acc=0.5682 val_auroc=0.7514 best(accuracy)=0.5682
[Epoch 007/40] loss=1.0710 val_acc=0.5267 val_auroc=0.7460 best(accuracy)=0.5682
[Epoch 008/40] loss=1.0767 val_acc=0.5564 val_auroc=0.7474 best(accuracy)=0.5682
[Epoch 009/40] loss=1.0671 val_acc=0.5623 val_auroc=0.7495 best(accuracy)=0.5682
[Epoch 010/40] loss=1.0541 val_acc=0.5579 val_auroc=0.7477 best(accuracy)=0.5682
[Epoch 011/40] loss=1.0578 val_acc=0.5653 val_auroc=0.7495 best(accuracy)=0.5682
[Epoch 012/40] loss=1.0543 val_acc=0.5386 val_au

,dataset,best_val_acc,test_acc,test_auroc,cf_validity,cf_changed_bits,cf_l1_cont,n_rules,n_binary_features
0,hyconex/adult,0.8071,0.8061,0.8735,0.7593,13.2435,13.2435,64,34
1,hyconex/audit,1.0000,1.0000,1.0000,0.0000,26.4918,2.1854,64,50
2,hyconex/blobs,0.9708,0.9433,0.9974,0.4717,3.9600,0.6322,24,8
3,hyconex/compas,0.5801,0.5083,0.7127,0.4157,11.4893,11.4893,64,25
4,hyconex/german_credit,0.6667,0.6417,0.7200,0.3583,40.3833,39.6333,64,74
5,hyconex/heloc,0.7307,0.7197,0.7846,0.3011,37.2484,5.0663,64,78
6,hyconex/law,0.7163,0.7770,0.8533,0.7995,5.0000,0.7570,24,12
7,hyconex/moons,0.9634,0.9805,0.9885,0.9805,4.4537,0.6892,24,8
8,hyconex/moons_with_blob,0.8938,0.8800,0.9662,0.7800,2.9200,0.6721,64,8
9,hyperlogic/brca-n,0.5000,0.4889,0.5000,0.5111,67.9778,0.0000,64,120



Jeux ignorés:
- hyconex/wine : list index out of range


## Helpers contrefactuels et règles

In [5]:
def flipped_literals(x_orig: np.ndarray, x_cf: np.ndarray, bin_names: list[str], max_show: int = 8) -> list[str]:
    diff = np.where(np.abs(x_orig - x_cf) > 1e-6)[0]
    out = []
    for j in diff[:max_show]:
        out.append(f"{bin_names[j]}: {int(x_orig[j])} → {int(x_cf[j])}")
    if len(diff) > max_show:
        out.append(f"… +{len(diff) - max_show} autres flips")
    return out


def build_cf_table(run: dict, n_examples: int = 5, seed: int = 42) -> pd.DataFrame:
    trainer = run['trainer']
    splits = run['splits']
    model = trainer.model
    assert model is not None

    x_test_bin = trainer.binarizer.transform(splits.X_test)
    x_test_t = torch.tensor(x_test_bin, dtype=torch.float32, device=trainer.device)
    with torch.no_grad():
        logits = model.predict_logits(x_test_t)
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()

    pool = np.where(y_pred == splits.y_test)[0]
    rng = np.random.default_rng(seed)
    n = min(n_examples, len(pool))
    idx = rng.choice(pool, size=n, replace=False) if n > 0 else np.array([], dtype=int)

    if idx.size == 0:
        return pd.DataFrame(columns=['sample', 'true', 'pred', 'target', 'valid_cf', 'n_flips', 'flips'])

    x_sub = torch.tensor(x_test_bin[idx], dtype=torch.float32, device=trainer.device)
    y_pred_orig = torch.tensor(y_pred[idx], dtype=torch.long, device=trainer.device)

    with torch.no_grad():
        x_cf_all, logits_cf_all = model.generate_counterfactuals_all_classes(x_sub, y_pred_orig)
        y_pred_cf = torch.argmax(logits_cf_all, dim=2).cpu().numpy()

    x_sub_np = x_sub.cpu().numpy()
    x_cf_np = x_cf_all.cpu().numpy()
    bin_names = trainer.binarizer.binary_feature_names()
    class_names = splits.class_names

    rows = []
    for i, sample_idx in enumerate(idx):
        src = int(y_pred_orig[i].item())
        for target in range(len(class_names)):
            if target == src:
                continue
            x_cf = x_cf_np[i, target]
            rows.append({
                'sample': int(sample_idx),
                'true': class_names[int(splits.y_test[sample_idx])],
                'pred': class_names[src],
                'target': class_names[target],
                'valid_cf': int(y_pred_cf[i, target] == target),
                'n_flips': int(np.sum(np.abs(x_sub_np[i] - x_cf) > 1e-6)),
                'flips': '; '.join(flipped_literals(x_sub_np[i], x_cf, bin_names)),
            })
    return pd.DataFrame(rows)


def rules_dataframe(run: dict, top_n: int = 10) -> pd.DataFrame:
    rules = run['rules'][:top_n]
    df = pd.DataFrame(rules)
    if 'if' in df.columns:
        df['if'] = df['if'].apply(lambda conds: ' AND\n'.join(conds) if isinstance(conds, list) else conds)
    return df

## Par jeu : contrefactuels et règles

In [6]:
for ds_key, run in runs.items():
    display(Markdown(f"### `{ds_key}` — contrefactuels"))
    cf_df = build_cf_table(run, n_examples=3)
    with pd.option_context('display.max_colwidth', 120, 'display.max_rows', 30):
        display(cf_df)

    display(Markdown(f"### `{ds_key}` — règles HyperLogic (top 10)"))
    rdf = rules_dataframe(run, top_n=10)
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None, 'display.width', None):
        display(rdf)

### `hyconex/adult` — contrefactuels

,sample,true,pred,target,valid_cf,n_flips,flips
0,4257,0,0,1,1,15,"age_bin0_[0.000,0.151): 1 → -1; age_bin2_[0.274,0.411): -1 → 1; age_bin3_[0.411,1.000): -1 → 1; hours_per_week_bin2_..."
1,600,0,0,1,1,17,"age_bin0_[0.000,0.151): 1 → -1; age_bin2_[0.274,0.411): -1 → 1; age_bin3_[0.411,1.000): -1 → 1; hours_per_week_bin0_..."
2,5045,0,0,1,1,11,"age_bin0_[0.000,0.151): 1 → -1; age_bin2_[0.274,0.411): -1 → 1; age_bin3_[0.411,1.000): -1 → 1; hours_per_week_bin2_..."


### `hyconex/adult` — règles HyperLogic (top 10)

,rule_id,if,then_class,score
0,41,marital_status=Single=+1 AND\neducation=Prof-school=-1 AND\nmarital_status=Married=-1 AND\neducation=HS-grad=-1,0,1.000000
1,36,"occupation=Blue-Collar=-1 AND\nmarital_status=Separated=+1 AND\nage_bin3_[0.411,1.000)=-1 AND\ngender=Female=+1",0,0.997995
2,26,"race=White=+1 AND\nage_bin1_[0.151,0.274)=+1 AND\nworkclass=Self-Employed=-1 AND\nworkclass=Private=-1",1,0.997547
3,4,"age_bin2_[0.274,0.411)=-1 AND\nage_bin0_[0.000,0.151)=-1 AND\neducation=Doctorate=+1 AND\ngender=Male=-1",0,0.995923
4,5,"age_bin3_[0.411,1.000)=+1 AND\nage_bin0_[0.000,0.151)=-1 AND\nrace=White=+1 AND\nworkclass=Government=-1",1,0.994946
5,1,"gender=Male=-1 AND\neducation=School=-1 AND\nage_bin1_[0.151,0.274)=-1 AND\neducation=Prof-school=-1",0,0.991078
6,16,education=Masters=-1 AND\neducation=HS-grad=-1 AND\noccupation=Sales=-1 AND\neducation=Bachelors=-1,1,0.989506
7,32,"race=White=+1 AND\nhours_per_week_bin0_[0.000,0.398)=+1 AND\nworkclass=Government=-1 AND\ngender=Male=-1",0,0.984735
8,25,race=White=+1 AND\nworkclass=Self-Employed=-1 AND\neducation=Masters=-1 AND\nmarital_status=Separated=-1,1,0.978475
9,61,"occupation=Sales=-1 AND\nage_bin3_[0.411,1.000)=+1 AND\nhours_per_week_bin1_[0.398,0.449)=-1 AND\neducation=Doctorate=-1",0,0.974959


### `hyconex/audit` — contrefactuels

...

,rule_id,if,then_class,score
0,38,"age_bin2_[0.676,0.824)=+1 AND\nap_lo_bin1_[0.014,0.014)=-1 AND\nap_hi_bin2_[0.018,1.000)=+1 AND\nweight_bin2_[0.323,0.376)=+1",1,0.999999
1,34,"gender=2=-1 AND\nage_bin0_[0.000,0.529)=-1 AND\ngluc=2=-1 AND\nap_hi_bin2_[0.018,1.000)=-1",0,0.999883
2,15,"ap_hi_bin2_[0.018,1.000)=+1 AND\nheight_bin2_[0.560,0.585)=+1 AND\nap_lo_bin0_[0.000,0.014)=+1 AND\ngluc=3=-1",0,0.999420
3,61,"gluc=2=+1 AND\nap_lo_bin0_[0.000,0.014)=-1 AND\nage_bin0_[0.000,0.529)=+1 AND\ncholesterol=1=-1",0,0.999083
4,37,"smoke=0=-1 AND\nalco=1=+1 AND\nheight_bin0_[0.000,0.528)=+1 AND\nactive=1=-1",0,0.999070
5,28,"height_bin3_[0.585,1.000)=-1 AND\ngluc=1=-1 AND\ngluc=3=+1 AND\nap_hi_bin0_[0.000,0.017)=+1",1,0.997808
6,62,"smoke=1=+1 AND\nap_lo_bin1_[0.014,0.014)=+1 AND\ncholesterol=2=+1 AND\nap_hi_bin0_[0.000,0.017)=+1",1,0.997611
7,6,"height_bin0_[0.000,0.528)=-1 AND\ncholesterol=1=-1 AND\ncholesterol=2=+1 AND\nage_bin0_[0.000,0.529)=+1",0,0.996992
8,17,"gender=1=-1 AND\ncholesterol=2=+1 AND\nap_lo_bin2_[0.014,1.000)=-1 AND\nap_hi_bin1_[0.017,0.018)=+1",0,0.994629
9,30,"ap_hi_bin1_[0.017,0.018)=-1 AND\nactive=0=+1 AND\nap_lo_bin1_[0.014,0.014)=-1 AND\nap_lo_bin2_[0.014,1.000)=+1",0,0.993584


### `hyperlogic/disease` — contrefactuels

,sample,true,pred,target,valid_cf,n_flips,flips
0,644,39,39,0,0,60,symptom=acute_liver_failure: -1 → 1; symptom=anxiety: -1 → 1; symptom=back_pain: -1 → 1; symptom=belly_pain: -1 → 1;...
1,644,39,39,1,0,59,symptom=acute_liver_failure: -1 → 1; symptom=anxiety: -1 → 1; symptom=back_pain: -1 → 1; symptom=belly_pain: -1 → 1;...
2,644,39,39,2,0,58,symptom=acute_liver_failure: -1 → 1; symptom=anxiety: -1 → 1; symptom=back_pain: -1 → 1; symptom=belly_pain: -1 → 1;...
3,644,39,39,3,0,59,symptom=acute_liver_failure: -1 → 1; symptom=anxiety: -1 → 1; symptom=back_pain: -1 → 1; symptom=belly_pain: -1 → 1;...
4,644,39,39,4,0,60,symptom=acute_liver_failure: -1 → 1; symptom=anxiety: -1 → 1; symptom=back_pain: -1 → 1; symptom=belly_pain: -1 → 1;...
...,...,...,...,...,...,...,...
115,760,3,3,36,0,58,symptom=abnormal_menstruation: -1 → 1; symptom=belly_pain: -1 → 1; symptom=bladder_discomfort: -1 → 1; symptom=blood...
116,760,3,3,37,0,60,symptom=abnormal_menstruation: -1 → 1; symptom=acidity: -1 → 1; symptom=belly_pain: -1 → 1; symptom=bladder_discomfo...
117,760,3,3,38,0,62,symptom=abnormal_menstruation: -1 → 1; symptom=acidity: -1 → 1; symptom=belly_pain: -1 → 1; symptom=bladder_discomfo...
118,760,3,3,39,0,56,symptom=abnormal_menstruation: -1 → 1; symptom=belly_pain: -1 → 1; symptom=bladder_discomfort: -1 → 1; symptom=blood...


### `hyperlogic/disease` — règles HyperLogic (top 10)

,rule_id,if,then_class,score
0,73,symptom=dark_urine=-1 AND\nsymptom=acute_liver_failure=-1 AND\nsymptom=yellowish_skin=-1 AND\nsymptom=blurred_and_distorted_vision=-1,7,0.371347
1,5,symptom=phlegm=-1 AND\nsymptom=puffy_face_and_eyes=-1 AND\nsymptom=depression=+1 AND\nsymptom=coma=-1,11,0.276325
2,71,symptom=mood_swings=-1 AND\nsymptom=extra_marital_contacts=+1 AND\nsymptom=enlarged_thyroid=+1 AND\nsymptom=brittle_nails=-1,16,0.259902
3,74,symptom=increased_appetite=-1 AND\nsymptom=cold_hands_and_feets=+1 AND\nsymptom=back_pain=+1 AND\nsymptom=prominent_veins_on_calf=-1,13,0.212892
4,68,symptom=dehydration=-1 AND\nsymptom=chills=+1 AND\nsymptom=cold_hands_and_feets=-1 AND\nsymptom=drying_and_tingling_lips=+1,36,0.210217
5,4,symptom=prominent_veins_on_calf=+1 AND\nsymptom=nausea=-1 AND\nsymptom=dehydration=-1 AND\nsymptom=pain_behind_the_eyes=+1,15,0.202299
6,7,symptom=watering_from_eyes=-1 AND\nsymptom=receiving_unsterile_injections=-1 AND\nsymptom=anxiety=-1 AND\nsymptom=nausea=+1,23,0.199088
7,32,symptom=coma=+1 AND\nsymptom=inflammatory_nails=+1 AND\nsymptom=weakness_in_limbs=+1 AND\nsymptom=phlegm=-1,20,0.190549
8,9,symptom=sinus_pressure=-1 AND\nsymptom=lethargy=-1 AND\nsymptom=dark_urine=+1 AND\nsymptom=vomiting=-1,22,0.182576
9,69,symptom=foul_smell_of urine=+1 AND\nsymptom=bruising=-1 AND\nsymptom=brittle_nails=+1 AND\nsymptom=pain_in_anal_region=-1,21,0.172180


## Exemple de règle complète (premier jeu entraîné)

In [7]:
if runs:
    first = next(iter(runs.values()))
    if first['rules']:
        rule = first['rules'][0]
        print('Jeu =', first['key'])
        print('rule_id =', rule.get('rule_id'))
        print('then_class =', rule.get('then_class'))
        print('score =', rule.get('score'))
        print('\nIF')
        for cond in rule.get('if', []):
            print(' ', cond)
        print('THEN →', rule.get('then_class'))

Jeu = hyconex/adult
rule_id = 41
then_class = 0
score = 1.0

IF
  marital_status=Single=+1
  education=Prof-school=-1
  marital_status=Married=-1
  education=HS-grad=-1
THEN → 0
